In [1]:
from agents import Agent, Runner, RunContextWrapper, WebSearchTool, function_tool, handoff, SQLiteSession, ItemHelpers
from itertools import islice
from typing import Iterator
from pydantic import BaseModel
import numpy as np
import pandas as pd
import json
import datetime
import os
import re
from openai import OpenAI

In [2]:
os.environ["OPENAI_API_KEY"] = (
    "***REMOVED***"
)

### Guardrail agents
* Category verification: "Accept" or "Flag" each categorization. Be strict with acceptances
* update query verification (TODO guardrails should be able to work on intermediate tool call steps)

## Context

In [6]:
BONUS_THRESH = 4_000
START_BALANCE = 10896.91
START_YEAR, START_MONTH, START_DAY = 2022, 9, 1

In [ ]:
@function_tool
def build_transactions_table_sql(context: RunContextWrapper[TransactionsAnalysisContextSQL]) -> str:
    """
    Build transactions table in SQL database from CSV files
    """
    # Clear existing transactions
    # context.context.execute_update("DELETE FROM transactions")

    # Load and process data (using your existing pandas logic for now)
    def reformat_date(dash_date: str) -> str:  # from "yyyy-mm-dd" to "mm/dd/yyyy"
        yyyy, mm, dd = dash_date.split("-")
        return "/".join([mm, dd, yyyy])

    # Load cc tables (same pandas logic for file processing)
    credit_chase_dfs = [
        pd.read_csv(f"{context.context.source_data_path}chase/{path}")
        for path in os.listdir(f"{context.context.source_data_path}/chase/")
        if path.startswith("Chase3098")
    ]
    for i in range(len(credit_chase_dfs)):
        credit_chase_dfs[i]["Account"] = "Sapphire Preferred"

    credit_capital_one_dfs = [
        pd.read_csv(f"{context.context.source_data_path}capital_one/{path}")
        for path in os.listdir(f"{context.context.source_data_path}/capital_one/")
    ]
    for i in range(len(credit_capital_one_dfs)):
        credit_capital_one_dfs[i]["Posted Date"] = credit_capital_one_dfs[i]["Posted Date"].apply(reformat_date)
        credit_capital_one_dfs[i]["Amount"] = -credit_capital_one_dfs[i]["Debit"].combine_first(
            -credit_capital_one_dfs[i]["Credit"]
        )
        credit_capital_one_dfs[i] = credit_capital_one_dfs[i].rename({"Posted Date": "Post Date"}, axis="columns")[
            ["Post Date", "Description", "Category", "Amount"]
        ]
        credit_capital_one_dfs[i]["Account"] = "VentureX"

    credit_df = pd.concat(credit_chase_dfs + credit_capital_one_dfs).reset_index(drop=True)[
        ["Account", "Post Date", "Description", "Amount"]
    ]
    credit_df = credit_df.sort_values("Post Date", ascending=True, ignore_index=True)
    credit_df = credit_df.rename({"Post Date": "Posting Date"}, axis="columns")
    credit_df["Posting Date"] = pd.to_datetime(credit_df["Posting Date"])

    # Load checking tables
    checking_dfs = [
        pd.read_csv(f"{context.context.source_data_path}chase/{path}", index_col=False)
        for path in os.listdir(f"{context.context.source_data_path}chase/")
        if path.startswith("Chase7113_Activity_")
    ]

    # Process checking data (same logic as before)
    max_dates = []
    for df in checking_dfs:
        max_dates.append(pd.to_datetime(df["Posting Date"]).max())

    dfs_to_concat = [checking_dfs[0]]
    for i in range(1, len(checking_dfs)):
        prev_max = max_dates[i - 1]
        curr_df = checking_dfs[i]
        filtered = curr_df.loc[pd.to_datetime(curr_df["Posting Date"]) > prev_max]
        dfs_to_concat.append(filtered)

    checking_df = pd.concat(dfs_to_concat, ignore_index=True)
    checking_df["Posting Date"] = pd.to_datetime(checking_df["Posting Date"])
    checking_df["Account"] = "Chase Checking"
    checking_df = checking_df.sort_values("Posting Date", ascending=True, ignore_index=True)
    checking_df = checking_df[["Account", "Posting Date", "Description", "Amount"]]

    transactions_df = pd.concat([credit_df, checking_df], ignore_index=True)

    # Filter from start date
    transactions_df = transactions_df[
        transactions_df["Posting Date"] >= datetime.datetime(day=START_DAY, month=START_MONTH, year=START_YEAR)
    ]

    # Split paychecks into base + others
    transactions_df = separate_paycheck_bonuses(transactions_df)

    # Add start balance
    start_row = pd.Series(
        {
            "Account": "Chase Checking",
            "Posting Date": datetime.datetime(day=START_DAY, month=START_MONTH, year=START_YEAR),
            "Description": "Start Balance",
            "Amount": START_BALANCE,
        }
    )
    transactions_df = pd.concat([pd.DataFrame([start_row]), transactions_df], ignore_index=True)
    transactions_df = transactions_df.sort_values(["Posting Date", "Description"], ascending=True, ignore_index=True)

    # Create transaction keys
    transactions_df["transaction_key"] = transactions_df.apply(
        lambda row: re.sub(r"\s+", " ", f"{row.name}_{row.Description}"), axis=1
    )

    # Load existing categories TODO sql version
    categories_dict = {}

    # Insert into SQL database
    insert_query = """
        INSERT OR REPLACE INTO transactions 
        (transaction_key, account, posting_date, description, amount, category)
        VALUES (?, ?, ?, ?, ?, ?)
    """

    # Prepare all rows for bulk insert as a single query
    values = []
    for _, row in transactions_df.iterrows():
        category = categories_dict.get(row["transaction_key"], None)
        if row["Description"] == "Start Balance":
            category = "N/A"
        values.append(
            (
                row["transaction_key"],
                row["Account"],
                row["Posting Date"].strftime("%Y-%m-%d"),
                row["Description"],
                float(row["Amount"]),
                category,
            )
        )

    # Build a single INSERT OR REPLACE statement with multiple VALUES
    placeholders = ",".join(["(?, ?, ?, ?, ?, ?)"] * len(values))
    bulk_insert_query = f"""
        INSERT OR REPLACE INTO transactions 
        (transaction_key, account, posting_date, description, amount, category)
        VALUES {placeholders}
    """
    # Flatten the values list for the query parameters
    flat_values = [item for row in values for item in row]
    context.context.connection.execute(bulk_insert_query, flat_values)
    context.context.connection.commit()

    return f"Loaded {len(transactions_df)} transactions into SQL database"


@function_tool
def get_uncategorized_transactions(context: RunContextWrapper[TransactionsAnalysisContextSQL]) -> str:
    """
    Get uncategorized transactions from SQL database
    """
    query = """
        SELECT transaction_key, description, amount, posting_date
        FROM transactions 
        WHERE category IS NULL OR category = ''
        ORDER BY posting_date DESC
    """

    uncategorized_df = context.context.execute_query(query)

    # Convert to iterator for compatibility
    transactions = [
        {"transaction_key": row["transaction_key"], "amount": row["amount"]} for _, row in uncategorized_df.iterrows()
    ]
    context.context.uncategorized_transactions = iter(transactions)

    return f"Found {len(uncategorized_df)} uncategorized transactions"





@function_tool
def get_category_definitions(context: RunContextWrapper[TransactionsAnalysisContextSQL]) -> str:
    """
    Get category definitions from SQL database
    """
    query = "SELECT category_name, definition FROM category_definitions ORDER BY category_name"
    results_df = context.context.execute_query(query)

    categories = {row["category_name"]: row["definition"] for _, row in results_df.iterrows()}
    return json.dumps(categories, separators=(",", ":"), indent=2)

## Tool implementations

In [8]:
def separate_paycheck_bonuses(transactions_df: pd.DataFrame) -> pd.DataFrame:
    """
    Separate paycheck transactions into separate rows for base and bonus/stock/etc., modifying transactions_df in place.
    """
    # Find Microsoft paychecks directly from transactions_df
    paycheck_mask = (transactions_df["Amount"] > 0) & (
        transactions_df.Description.str.contains("MICROSOFT", case=False, na=False)
    )
    paycheck_df = transactions_df[paycheck_mask].copy()

    bonus_rows = []
    for i, (idx, row) in enumerate(paycheck_df.iterrows()):
        if row.Amount > BONUS_THRESH:
            # get bonus amount by comparing to surrounding paychecks
            if (i + 1 < len(paycheck_df)) and paycheck_df.iloc[i + 1].Amount > 2000:
                bonus_amt = row.Amount - paycheck_df.iloc[i + 1].Amount
            elif i != 0 and paycheck_df.iloc[i - 1].Amount > 2000:
                bonus_amt = row.Amount - paycheck_df.iloc[i - 1].Amount
            else:
                raise Exception("no valid paycheck to compare")

            # update original dataframe in place
            transactions_df.loc[idx, "Amount"] -= bonus_amt

            # create bonus row
            bonus_row = row.copy()
            bonus_row.Amount = bonus_amt
            bonus_row.Description = "Microsoft Bonus/Stock"
            bonus_rows.append(bonus_row)

    if bonus_rows:
        bonus_df = pd.DataFrame(bonus_rows)
        transactions_df_with_bonuses = pd.concat([transactions_df, bonus_df], ignore_index=True)
        transactions_df_with_bonuses.sort_values(
            ["Posting Date", "Description"], ascending=True, inplace=True, ignore_index=True
        )
        return transactions_df_with_bonuses
    else:
        return transactions_df

In [9]:
@function_tool
def get_next_uncategorized_transactions(context: RunContextWrapper[TransactionsAnalysisContextSQL], n: int = 10) -> str:
    """
    Gets the next transactions to categorize.
    """
    # get n items from the iterator
    context.context.current_uncategorized_transactions = [
        next(context.context.uncategorized_transactions) for _ in range(n)
    ]
    return context.context.current_uncategorized_transactions


@function_tool
def update_category_of_current_uncategorized_transactions(
    context: RunContextWrapper[TransactionsAnalysisContextSQL], categories: str
) -> str:
    """
    Updates the category of the current transaction in the transactions table.

    Args:
        context: Agent context containing transactions table
        categories: Categories to set for the current transactions; JSON mapping transaction keys to categories
    """

    if (
        not hasattr(context.context, "current_uncategorized_transactions")
        or len(context.context.current_uncategorized_transactions) == 0
    ):
        raise ValueError("No current uncategorized transactions to update.")

    categories_dict = json.loads(categories)

    update_query = """
        UPDATE transactions
        SET category = ?
        WHERE transaction_key = ?
    """
    successes = 0
    error_messages = []

    for transaction in context.context.current_uncategorized_transactions:
        transaction_key = transaction["transaction_key"]
        category = categories_dict.get(transaction_key, None)
        if category is None:
            error_messages.append(f"No category provided for transaction {transaction_key}")

        context.context.execute_update(update_query, (category, transaction_key))
        successes += 1

    return f"Updated categories for {successes} transactions; provided categories were {categories_dict}; error messages were {error_messages}"

In [10]:
@function_tool
def refresh_categorized_transaction_embeddings(context: RunContextWrapper[TransactionsAnalysisContextSQL]) -> str:
    """
    Refreshes the embeddings of the categorized transactions.

    Args:
        context: Agent context, including `transactions` table

    Returns:
        str: Number of categorized transactions
    """
    # 1. Get all categorized transactions
    categorized_transactions = context.context.execute_query(
        """
        SELECT transaction_key, description, category
        FROM transactions
        WHERE category IS NOT NULL AND category != ''
        ORDER BY posting_date DESC
        """
    )

    # 2. Get the embeddings of the transaction descriptions
    embedding_client = OpenAI()
    context.context.transaction_description_embeddings = {
        row["description"]: (
            embedding_client.embeddings.create(input=row["description"], model="text-embedding-3-small")
            .data[0]
            .embedding
        )
        for _, row in categorized_transactions.iterrows()
    }
    return f"Number of categorized transactions: {len(context.context.transaction_description_embeddings)}"

In [11]:
from openai import OpenAI

embedding_client = OpenAI()
categorized_embeddings = [
    d.embedding for d in embedding_client.embeddings.create(input=["hi", "bye"], model="text-embedding-3-small").data
]

In [12]:
embeddings_cache = {}


@function_tool
def get_transactions_category_examples(context: RunContextWrapper[TransactionsAnalysisContextSQL]):
    """






    Gets the category of the most similar categorized transaction to each of the uncategorized transactions.







    Returns:






        dict[str, str]: key = transaction description, value = transaction category
    """

    # Get uncategorized transactions from database

    uncategorized_query = """






        SELECT description






        FROM transactions 






        WHERE category IS NULL OR category = ''






        ORDER BY posting_date DESC






        LIMIT 10






    """

    uncategorized_transactions = context.context.execute_query(uncategorized_query)

    # Get categorized transactions for comparison

    categorized_query = """






        SELECT description, category






        FROM transactions






        WHERE category IS NOT NULL AND category != ''






    """

    categorized_transactions_df = context.context.execute_query(categorized_query)

    if categorized_transactions_df.empty:

        return "No categorized transactions available for examples"

    category_examples = {}

    embedding_client = OpenAI()
    print("computing embeddings for categorized transactions")
    categorized_descriptions = categorized_transactions_df["description"].tolist()

    # Compute embeddings for categorized transactions in two halves
    half = len(categorized_descriptions) // 2
    first_half_descriptions = categorized_descriptions[:half]
    second_half_descriptions = categorized_descriptions[half:]

    print("computing embeddings for first half of categorized transactions")
    first_half_embeddings = embedding_client.embeddings.create(
        input=first_half_descriptions, model="text-embedding-3-small"
    ).data
    for i, categorized_description in enumerate(first_half_descriptions):
        categorized_embedding = first_half_embeddings[i].embedding
        embeddings_cache[categorized_description] = categorized_embedding

    print("computing embeddings for second half of categorized transactions")
    second_half_embeddings = embedding_client.embeddings.create(
        input=second_half_descriptions, model="text-embedding-3-small"
    ).data
    for i, categorized_description in enumerate(second_half_descriptions):
        categorized_embedding = second_half_embeddings[i].embedding
        embeddings_cache[categorized_description] = categorized_embedding

    print("done")

    # For each uncategorized transaction, find the most similar categorized one

    # get embeddings for all uncategorized transactions
    print("computing embeddings for uncategorized transactions")
    uncategorized_descriptions = uncategorized_transactions["description"].tolist()
    uncategorized_embeddings = embedding_client.embeddings.create(
        input=uncategorized_descriptions, model="text-embedding-3-small"
    ).data
    for i, uncategorized_description in enumerate(uncategorized_descriptions):
        uncategorized_embedding = uncategorized_embeddings[i].embedding
        embeddings_cache[uncategorized_description] = uncategorized_embedding

    for i, uncategorized_row in uncategorized_transactions.iterrows():
        print(i)

        uncategorized_description = uncategorized_row["description"]

        assert uncategorized_description in embeddings_cache
        uncategorized_embedding = embeddings_cache[uncategorized_description]

        best_similarity = -1

        best_category = None

        # Compare with all categorized transactions

        for _, categorized_row in categorized_transactions_df.iterrows():

            categorized_description = categorized_row["description"]

            assert categorized_description in embeddings_cache
            categorized_embedding = embeddings_cache[categorized_description]

            # Calculate similarity

            similarity = np.dot(uncategorized_embedding, categorized_embedding)

            if similarity > best_similarity:

                best_similarity = similarity

                best_category = categorized_row["category"]

        if best_category:

            category_examples[categorized_description] = best_category

    # Save examples in context for validation agent

    context.context.uncategorized_transactions_category_examples = category_examples

    # save cache
    with open("embeddings_cache.json", "w") as f:
        json.dump(embeddings_cache, f)

    # Return as JSON string

    return json.dumps(category_examples, indent=2)

In [14]:
from enum import Enum

ValidationResult = Enum("ValidationResult", ["Accept", "Flag"])

## Agent implementations

In [15]:
def category_verification_agent_instructions(category_examples: dict[str, str]) -> str:  # TODO
    """
    Instructions for the category verification agent.
    """
    return (
        "You are the Category Verification Agent. You validate the categories assigned to transactions against the category definitions, and flag any transactions that do not clearly fir their assigned category.\n"
        "Do the following:\n"
        "1. Invoke `get_category_definitions`.\n"
        f"2. Align with the following ground-truth category examples: {category_examples}\n"
        "3. For each of the following transactions, output `Accept` if the category is clearly correct, or `Flag` otherwise (so that it can be manually verified).\n"
    )

In [16]:
def transaction_categorizer_instructions() -> str:
    """
    Instructions for the transaction categorizer agent.

    Returns:
        str: Instructions for the agent.
    """
    return (
        f"You are the Transaction Categorizer Agent, part of the Transactions Analysis agent system. Your job is to categorize each transaction you are given for analysis.\n"
        "When you are invoked, take the following steps:\n"
        "1. Get transaction category definitions using `get_category_definitions`\n"
        "2. Get examples of categorized transactions using `get_transactions_category_examples`\n"
        "3. Compile all uncategorized transactions using `get_uncategorized_transactions`\n"
        "4. while `get_next_uncategorized_transactions` returns transactions:\n"
        "4.1. Get the next group of transactions to categorize using `get_next_uncategorized_transactions` with n=6\n"
        "4.2. Record the best categories for your new transactions by calling `update_category_of_current_uncategorized_transactions`\n"
        "    * `categories` is a JSON mapping each `transaction_key` from `get_next_uncategorized_transactions` to a category from `get_category_definitions`\n"
        "    * Provide each `transaction_key` with the exact same whitespace as in `get_next_uncategorized_transactions`, so that they can be matched\n"
        "4.3. Repeat until `get_next_uncategorized_transactions` returns no transactions\n"
        "5. Verify that you're done by calling `get_uncategorized_transactions` and checking that it returns no uncategorized transactions.\n"
        "6. Return 'Done!'"
    )

In [ ]:
def transaction_table_refresher_instructions() -> str:  # TODO
    """
    Instructions for the table refresher agent.

    Returns:
        A string containing the instructions for the table updater agent.
    """
    return (
        "You are the Transactions Table Refresher Agent, part of the Transactions Analysis agent system. Your role is to refresh the transactions table to include the latest processed transactions.\n"
        "When you are invoked, perform the following steps:\n"
        "1. Build the latest transactions table using `refresh_transactions_table`.\n"  # TODO
        "3. Hand off to the Transaction Categorizer agent to categorize any new transactions.\n"
        '4. Finally, return "Done refreshing transactions table!" to indicate that the transactions table has been successfully refreshed.\n'
    )




def advisor_agent_instructions() -> str:
    """
    Instructions for the advisor agent.

    Returns:
        A string containing the instructions for the advisor agent.
    """
    return (
        "You are the Advisor Agent, part of the Transactions Analysis agent system. You have been invoked because the user has requested financial advice that cannot be provided solely using the transactions table.\n"
        "\n"
        f"First, get the schema of the transactions table for your own reference by invoking `get_transactions_table_schema`.\n"
        "\n"
        "Then, answer the user's query using any of the following:\n"
        "- The Query Answering Agent (query-answering-agent), to which you can provide your own queries about the transactions table\n"
        "- The search_web tool, which you can use to look online for general financial advice based on the user's circumstances\n"
        "- The `query_transactions_table` tool, to which you can provide simple SQL queries.\n"
        "   - Complex queries should be handled by the Query Answering Agent.\n"
        "- The `plot` tool, which you can use to draw plots with Matplotlib\n"
        "\n"
        "Once you have enough information to answer the user's question, provide the answer as your final response."
    )


def orchestrator_agent_instructions() -> str:  # TODO
    """
    Instructions for the top-level Transactions Analysis agent.

    Returns:
        A string containing the instructions for the top-level agent.
    """

    return (
        "You are the Transactions Analysis agent. You track, categorize, and analyze the user's individual debit and credit transactions to help them better understand their spending.\n"
        "At the beginning of your conversation, invoke the Transactions Table Refresher Agent to update the table with the user's latest transactions.\n"
        "After the table is refreshed, invoke `get_transactions_table_schema` so that you know what information is available about each transaction.\n"
        "Next, inform the user that you can help with the following:\n"
        "\t1) Providing a report summarizing the user's last month of spending\n"
        "\t2) Answering exploratory questions about the user's transactions\n"
        "\t\t- Give two examples of useful things the user might want to know about their transactions; prioritize creative examples the user might not have thought of.\n"
        "\t3) Providing financial advice from the web based on the user's transactions\n"
        "\t4) Refreshing the transactions table again if the user adds new transactions data\n"
        "Now that you're ready, answer the user's queries by handing each one off to one of the following agents/tools:\n"
        "\t- `generate_monthly_report`: A tool to summarize the user's last month of spending\n"
        "\t- `Query Answering Agent`: An agent that answers exploratory questions about the transactions table\n"
        "\t- `Financial Advisor Agent`: An agent that searches the web to give the user advice based on their transactions\n"
        "\t- `Transactions Table Refresher Agent`: An agent that refreshes the transactions table based on newly-added transactions data."
    )

## Run agent

In [18]:
context = TransactionsAnalysisContextSQL("data/")
# await build_transactions_table_sql.on_invoke_tool(RunContextWrapper(context), "")

In [ ]:
from openai.types.responses import ResponseTextDeltaEvent
import time


In [40]:
await run_agent(
    query_answering_agent,
    "What grocery store have i been to the most? and how much have i spent there?",
)

Agent updated: query-answering-agent
-- Tool was called: ResponseFunctionToolCall(arguments='{"table_name":"transactions"}', call_id='call_Fs1SslCeTcla4qqGc9mpOTXD', name='get_table_schema', type='function_call', id='fc_68792a6ae3c4819a8cf77e6004b984ec0ab3b786a92a11f3', status='completed')
-- Tool output: transactions table schema:
- transaction_key: TEXT (Primary Key)
- account: TEXT
- posting_date: DATE
- description: TEXT
- amount: REAL
- category: TEXT (Values: N/A, Shopping, Solo necessary meals, Social food/drinks, Rent, Grocery, Transit, Misc, Bills, Entertainment, Venmo/ATM, Subscriptions, Income, Travel, Exercise, Investments, Medical, Donations)

-- Tool was called: ResponseFunctionToolCall(arguments='{"sql_query":"SELECT description AS merchant, COUNT(*) AS visit_count, SUM(amount) AS total_spent\\nFROM transactions\\nWHERE category = \'Grocery\'\\nGROUP BY merchant\\nORDER BY visit_count DESC\\nLIMIT 1;","target_table_name":""}', call_id='call_w7gXfBJMgcdBGC91ez5ewL37', nam

In [34]:
await get_uncategorized_transactions.on_invoke_tool(RunContextWrapper(context), "")

'Found 1 uncategorized transactions'

In [69]:
transaction_categorizer_agent = Agent(
    name="transaction-categorizer-agent",
    instructions=transaction_categorizer_instructions(),
    tools=[
        get_next_uncategorized_transactions,
        get_category_definitions,
        get_transactions_category_examples,
        update_category_of_current_uncategorized_transactions,
        get_uncategorized_transactions,
    ],
)

In [ ]:



while len(list(context.uncategorized_transactions)) > 0:
    try:
        session = await run_agent(transaction_categorizer_agent)
    except Exception as e:
        print(f"Error running agent: {e}")
        # If an error occurs, we wait to avoid throttling
    print("Waiting to avoid throttling...")
    time.sleep(60)

Agent updated: transaction-categorizer-agent
computing embeddings for categorized transactions
computing embeddings for first half of categorized transactions
computing embeddings for second half of categorized transactions
done
computing embeddings for uncategorized transactions
0
-- Tool was called
-- Tool was called
-- Tool was called
-- Tool output: {
  "Bills":"Mandatory, recurring payments",
  "Clothing":"Purchases at clothing stores / that are probably of clothing",
  "Credit card payments":"Payments made to credit card accounts",
  "Entertainment":"Concerts, sports games, movies, etc.",
  "Exercise":"Gym, sports leagues, etc.",
  "Grocery":"Payments at grocery stores",
  "Income":"Money received from work, investments, or other sources",
  "Investments":"Transactions with investment accounts",
  "Medical":"Medical expenses",
  "Misc":"Transactions that don't fit into any other category",
  "Shopping":"Non-clothing shopping",
  "Social food/drinks":"Restaurants / bars with frien

In [45]:
c = json.loads(
    '{"categories":"{\\"3220_MICROSOFT        EDIPAYMENT                 PPD ID: 9911144442\\":\\"Subscriptions\\",\\"3216_Celebrity Series\\":\\"Entertainment\\",\\"3217_VENMO            PAYMENT    1039263652407   WEB ID: 3264681992\\":\\"Venmo/ATM\\",\\"3215_CHICK-FIL-A #00862\\":\\"Solo necessary meals\\",\\"3212_CAMBRIDGE HEALTH MYCHART\\":\\"Medical\\",\\"3213_PHR*FondrenOrthopedicGrou\\":\\"Medical\\",\\"3214_ROBINHOOD        CREDITS                    PPD ID: 5321710001\\":\\"Investments\\",\\"3210_PASADENA MRI & DIAGNOSTIC\\":\\"Medical\\",\\"3211_REMOTE ONLINE DEPOSIT #          1\\":\\"Income\\",\\"3206_AMAZON RETA* ZE3KJ5TM2\\":\\"Shopping\\"}"}'
)

c["categories"]

'{"3220_MICROSOFT        EDIPAYMENT                 PPD ID: 9911144442":"Subscriptions","3216_Celebrity Series":"Entertainment","3217_VENMO            PAYMENT    1039263652407   WEB ID: 3264681992":"Venmo/ATM","3215_CHICK-FIL-A #00862":"Solo necessary meals","3212_CAMBRIDGE HEALTH MYCHART":"Medical","3213_PHR*FondrenOrthopedicGrou":"Medical","3214_ROBINHOOD        CREDITS                    PPD ID: 5321710001":"Investments","3210_PASADENA MRI & DIAGNOSTIC":"Medical","3211_REMOTE ONLINE DEPOSIT #          1":"Income","3206_AMAZON RETA* ZE3KJ5TM2":"Shopping"}'

In [36]:
items = await session.get_items()

In [37]:
items

[{'content': 'Hi! Get started', 'role': 'user'},
 {'arguments': '{}',
  'call_id': 'call_rV6saw0Mhc6WqoPvIyoTjOTF',
  'name': 'get_category_definitions',
  'type': 'function_call',
  'id': 'fc_6875b41511bc8198a8df39606f7e31820b6dcd541f62540e',
  'status': 'completed'},
 {'arguments': '{}',
  'call_id': 'call_R96Y2uxt8JjkHPuKAWcHlRiL',
  'name': 'get_uncategorized_transactions',
  'type': 'function_call',
  'id': 'fc_6875b41533cc8198a0bca817ee1b685b0b6dcd541f62540e',
  'status': 'completed'},
 {'call_id': 'call_rV6saw0Mhc6WqoPvIyoTjOTF',
  'output': '{\n  "Bills":"Mandatory, recurring payments",\n  "Clothing":"Purchases at clothing stores / that are probably of clothing",\n  "Credit card payments":"Payments made to credit card accounts",\n  "Entertainment":"Concerts, sports games, movies, etc.",\n  "Exercise":"Gym, sports leagues, etc.",\n  "Grocery":"Payments at grocery stores",\n  "Income":"Money received from work, investments, or other sources",\n  "Investments":"Transactions with i

In [ ]:
# set transaction "3220_MICROSOFT EDIPAYMENT PPD ID: 9911144442" to "Income"

In [ ]:
session = SQLiteSession("test")
r = await Runner.run(
    starting_agent=transaction_categorizer_agent,
    context=context,
    input="Hi! Get started",
    session=session,
    max_turns=1,
)

In [49]:
await get_next_uncategorized_transactions.on_invoke_tool(RunContextWrapper(context), "")

['3248_AMAZON MARK* ZD8K13XK0',
 '3249_AMAZON MARK* ZP9GJ7RP1',
 '3250_BROTHERS MARKETPLACE #404',
 '3251_SADDLEBACK MOUNTAIN ECOMM',
 '3252_SADDLEBACK MOUNTAIN ECOMM',
 '3253_VENMO            PAYMENT    1039507216806   WEB ID: 3264681992',
 '3242_AIRBNB * HMN24PFAW3',
 '3243_FID BKG SVC LLC  MONEYLINE                  PPD ID: 0368504603',
 '3244_Morgan Stanley   ACH CREDIT                 PPD ID: 9827837001',
 '3245_SQ *PEPITA COFFEE CO.']

In [60]:
await update_category_of_current_uncategorized_transactions.on_invoke_tool(
    RunContextWrapper(context),
    json.dumps(
        {
            "categories": json.dumps(
                {
                    "3248_AMAZON MARK* ZD8K13XK0": "Shopping",
                    "3249_AMAZON MARK* ZP9GJ7RP1": "Shopping",
                    "3250_BROTHERS MARKETPLACE #404": "Grocery",
                    "3251_SADDLEBACK MOUNTAIN ECOMM": "Entertainment",
                    "3252_SADDLEBACK MOUNTAIN ECOMM": "Entertainment",
                    "3253_VENMO            PAYMENT    1039507216806   WEB ID: 3264681992": "Venmo/ATM",
                    "3242_AIRBNB * HMN24PFAW3": "Travel",
                    "3243_FID BKG SVC LLC  MONEYLINE                  PPD ID: 0368504603": "Investments",
                    "3244_Morgan Stanley   ACH CREDIT                 PPD ID: 9827837001": "Investments",
                    "3245_SQ *PEPITA COFFEE CO.": "Social food/drinks",
                }
            )
        }
    ),
)

In [82]:
items = await session.get_items()
items

[]

In [ ]:
# Placeholder for additional agent definitions that reference undefined tools
# These would need the missing tools to be implemented

query_answering_agent = Agent(
    name="query-answering-agent", 
    instructions=query_answering_agent_instructions(),
    tools=[query_transactions_table, plot], # TODO plot = query to produce data, series names, labels
)

# advisor_agent = Agent(
#     name="advisor-agent",
#     instructions=advisor_agent_instructions(), 
#     tools=[query_transactions_table, plot, WebSearchTool],
#     handoffs=[handoff(query_answering_agent)],
# )